# proceso de entrenamiento y selección de modelos

In [7]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor

import joblib
import os
import mlflow

import numpy as np
import pandas as pd
import time
import logging

import mlflow
import mlflow.sklearn

from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

import time
import numpy as np
import mlflow
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error


print("Versión de sklearn:", __import__("sklearn").__version__)

Versión de sklearn: 1.6.1


In [9]:
ruta_data = "../data/interim/feature_exploration_scaled.csv"
df = pd.read_csv(ruta_data)

print("Shape del dataframe:", df.shape)
df.head()

Shape del dataframe: (2823, 41)


,ORDERDATE,CITY,PRODUCTLINE,STATUS,QUANTITYORDERED,PRICEEACH,SALES,CITY_TOP,CITY_TOP_Madrid,CITY_TOP_Manchester,...,STATUS_Shipped,SALES_LOG1P,PRICEEACH_LOG1P,QUANTITYORDERED_YJ,SALES_LOG1P_STD,PRICEEACH_LOG1P_STD,QUANTITYORDERED_YJ_STD,SALES_MM,PRICEEACH_MM,QUANTITYORDERED_MM
0,2003-01-06,Nashua,Vintage Cars,Shipped,30.0,100.000,5151.00,Other,0.0,0.0,...,1.0,8.547140,4.615121,10.779233,0.977215,0.744732,-0.502703,0.515174,1.000000,0.279486
1,2003-01-06,Nashua,Vintage Cars,Shipped,50.0,67.800,3390.00,Other,0.0,0.0,...,1.0,8.128880,4.231204,14.925497,0.161327,-0.604671,1.557932,0.302584,0.519159,0.838457
2,2003-01-06,Nashua,Vintage Cars,Shipped,22.0,86.510,1903.22,Other,0.0,0.0,...,1.0,7.551828,4.471753,8.805943,-0.964311,0.240819,-1.483401,0.123099,0.798554,0.055897
3,2003-01-06,Nashua,Vintage Cars,Shipped,49.0,34.470,1689.03,Other,0.0,0.0,...,1.0,7.432502,3.568687,14.736932,-1.197077,-2.933305,1.464218,0.097242,0.021444,0.810509
4,2003-01-09,Frankfurt,Vintage Cars,Shipped,45.0,33.034,1404.00,Other,0.0,0.0,...,1.0,7.247793,3.527360,13.966075,-1.557384,-3.078563,1.081113,0.062833,0.000000,0.698714


# 80/20

In [10]:
TARGET_COL = "SALES"   # ajusta el nombre si tu columna objetivo se llama distinto

y = df[TARGET_COL]
X = df.drop(columns=[TARGET_COL])

print("Shape X:", X.shape, " | Shape y:", y.shape)

Shape X: (2823, 40)  | Shape y: (2823,)


In [11]:
n_total = len(df)
n_train = int(n_total * 0.8)

X_train = X.iloc[:n_train].copy()
y_train = y.iloc[:n_train].copy()

X_val = X.iloc[n_train:].copy()
y_val = y.iloc[n_train:].copy()

print("Tamaño train:", X_train.shape, y_train.shape)
print("Tamaño validación:", X_val.shape, y_val.shape)

Tamaño train: (2258, 40) (2258,)
Tamaño validación: (565, 40) (565,)


# pipeline de features

In [12]:
ruta_feature_pipe = "./models/feature_pipeline.pkl"

assert os.path.exists(ruta_feature_pipe), f"No encuentro: {ruta_feature_pipe}"

feature_pipeline = joblib.load(ruta_feature_pipe)
print("Tipo de feature_pipeline:", type(feature_pipeline))

Tipo de feature_pipeline: <class 'sklearn.pipeline.Pipeline'>


# Modelos y configuraciones

In [16]:
model_configs = {
    "linear_regression": {
        "estimator": LinearRegression,
        "params": [
            {"fit_intercept": True,  "positive": False},
            {"fit_intercept": False, "positive": False},
            {"fit_intercept": True,  "positive": True},
        ],
    },
    "ridge": {
        "estimator": Ridge,
        "params": [
            {"alpha": 0.1},
            {"alpha": 1.0},
            {"alpha": 10.0},
        ],
    },
    "random_forest": {
        "estimator": RandomForestRegressor,
        "params": [
            {"n_estimators": 100, "max_depth": None,  "min_samples_split": 2, "random_state": 42},
            {"n_estimators": 200, "max_depth": 10,    "min_samples_split": 2, "random_state": 42},
            {"n_estimators": 300, "max_depth": 20,    "min_samples_split": 5, "random_state": 42},
        ],
    },
    "gradient_boosting": {
        "estimator": GradientBoostingRegressor,
        "params": [
            {"n_estimators": 100, "learning_rate": 0.1, "max_depth": 3, "random_state": 42},
            {"n_estimators": 200, "learning_rate": 0.05, "max_depth": 3, "random_state": 42},
            {"n_estimators": 200, "learning_rate": 0.1, "max_depth": 4, "random_state": 42},
        ],
    },
    "knn": {
        "estimator": KNeighborsRegressor,
        "params": [
            {"n_neighbors": 3,  "weights": "uniform"},
            {"n_neighbors": 5,  "weights": "distance"},
            {"n_neighbors": 10, "weights": "distance"},
        ],
    },
}

len(model_configs)

5

# Configuración de conexión con mlflow

In [17]:
mlflow.set_tracking_uri("http://127.0.0.1:8080/")
mlflow.set_experiment("Proyecto Final")

<Experiment: artifact_location='mlflow-artifacts:/721817035780238080', creation_time=1764376829827, experiment_id='721817035780238080', last_update_time=1764376829827, lifecycle_stage='active', name='Proyecto Final', tags={'mlflow.experimentKind': 'custom_model_development'}>

# Entrenamiento

In [18]:
resultados = []
best_rmse = np.inf
best_model_name = None
best_config = None
best_pipeline = None

start_time = time.time()  # para medir tiempo total de tuning

mlflow.set_experiment("Proyecto Final")  # o el nombre que ya estés usando

with mlflow.start_run(run_name="tuning_modelos_ventas"):
    for model_name, cfg in model_configs.items():
        EstimatorClass = cfg["estimator"]

        for i, param_dict in enumerate(cfg["params"], start=1):
            print(f"\nEntrenando {model_name} - config {i} con params: {param_dict}")

            model = EstimatorClass(**param_dict)

            full_pipeline = Pipeline(steps=[
                ("features", feature_pipeline),  # tu pipeline de features del notebook 03
                ("model", model),
            ])

            # Entrenamos en el 80% inicial
            full_pipeline.fit(X_train, y_train)

            # Validamos en el 20% final
            y_pred = full_pipeline.predict(X_val)
            rmse = np.sqrt(mean_squared_error(y_val, y_pred))
            print(f"→ RMSE validación: {rmse:0.4f}")

            resultados.append({
                "model": model_name,
                "config_id": i,
                "params": param_dict,
                "rmse_val": rmse,
            })

            # Log en MLflow
            mlflow.log_metric(f"rmse_{model_name}_cfg_{i}", float(rmse))
            for p_name, p_value in param_dict.items():
                mlflow.log_param(f"{model_name}_cfg_{i}_{p_name}", p_value)

            # Actualizar mejor modelo
            if rmse < best_rmse:
                best_rmse = rmse
                best_model_name = model_name
                best_config = param_dict
                best_pipeline = full_pipeline

    total_time = time.time() - start_time
    mlflow.log_metric("tiempo_total_tuning_segundos", float(total_time))

print("\n=== Mejor modelo encontrado ===")
print("Modelo:", best_model_name)
print("Hiperparámetros:", best_config)
print("RMSE validación:", best_rmse)


Entrenando linear_regression - config 1 con params: {'fit_intercept': True, 'positive': False}
→ RMSE validación: 0.0000

Entrenando linear_regression - config 2 con params: {'fit_intercept': False, 'positive': False}
→ RMSE validación: 350.6202

Entrenando linear_regression - config 3 con params: {'fit_intercept': True, 'positive': True}
→ RMSE validación: 0.0000

Entrenando ridge - config 1 con params: {'alpha': 0.1}
→ RMSE validación: 0.6100

Entrenando ridge - config 2 con params: {'alpha': 1.0}
→ RMSE validación: 5.5733

Entrenando ridge - config 3 con params: {'alpha': 10.0}
→ RMSE validación: 43.5375

Entrenando random_forest - config 1 con params: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'random_state': 42}
→ RMSE validación: 6.6898

Entrenando random_forest - config 2 con params: {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 2, 'random_state': 42}
→ RMSE validación: 6.0213

Entrenando random_forest - config 3 con params: {'n_estimators': 

In [19]:
resultados_df = pd.DataFrame(resultados)
resultados_df.sort_values(by="rmse_val", inplace=True)
resultados_df.reset_index(drop=True, inplace=True)
resultados_df

,model,config_id,params,rmse_val
0,linear_regression,3,"{'fit_intercept': True, 'positive': True}",1.169921e-12
1,linear_regression,1,"{'fit_intercept': True, 'positive': False}",4.469992e-12
2,ridge,1,{'alpha': 0.1},6.100347e-01
3,ridge,2,{'alpha': 1.0},5.573334e+00
4,random_forest,2,"{'n_estimators': 200, 'max_depth': 10, 'min_sa...",6.021305e+00
5,random_forest,1,"{'n_estimators': 100, 'max_depth': None, 'min_...",6.689791e+00
6,random_forest,3,"{'n_estimators': 300, 'max_depth': 20, 'min_sa...",7.664740e+00
7,gradient_boosting,2,"{'n_estimators': 200, 'learning_rate': 0.05, '...",8.575506e+00
8,gradient_boosting,3,"{'n_estimators': 200, 'learning_rate': 0.1, 'm...",9.326924e+00
9,gradient_boosting,1,"{'n_estimators': 100, 'learning_rate': 0.1, 'm...",1.566995e+01


In [20]:
print("Reentrenando el mejor modelo con todos los datos disponibles...")
best_pipeline.fit(X, y)

# Ruta donde guardaremos el pipeline completo
ruta_full_pipeline = "./models/full_sales_forecast_pipeline.pkl"

joblib.dump(best_pipeline, ruta_full_pipeline)
print("Pipeline completo guardado en:", ruta_full_pipeline)

Reentrenando el mejor modelo con todos los datos disponibles...
Pipeline completo guardado en: ./models/full_sales_forecast_pipeline.pkl


In [21]:
loaded_pipeline = joblib.load(ruta_full_pipeline)

# Tomamos las últimas 5 filas como ejemplo
X_sample = X.tail(5)
y_true_sample = y.tail(5)

y_pred_sample = loaded_pipeline.predict(X_sample)

res_prueba = pd.DataFrame({
    "y_real": y_true_sample.values,
    "y_pred": y_pred_sample,
})
res_prueba

,y_real,y_pred
0,1895.94,1895.94
1,4692.60,4692.60
2,5894.94,5894.94
3,2702.04,2702.04
4,3777.58,3777.58


In [22]:
import time
import mlflow
import mlflow.sklearn

In [23]:
mlflow.set_experiment("Proyecto Final")

<Experiment: artifact_location='mlflow-artifacts:/721817035780238080', creation_time=1764376829827, experiment_id='721817035780238080', last_update_time=1764376829827, lifecycle_stage='active', name='Proyecto Final', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [24]:
resultados = []
best_rmse = np.inf
best_model_name = None
best_config = None
best_pipeline = None

start_time = time.time()

with mlflow.start_run(run_name="tuning_modelos_ventas") as parent_run:
    # --- loop de modelos y configs (CHALLENGERS) ---
    for model_name, cfg in model_configs.items():
        EstimatorClass = cfg["estimator"]

        for config_id, param_dict in enumerate(cfg["params"], start=1):
            print(f"\nEntrenando {model_name} - config {config_id} con params: {param_dict}")

            # nuevo run hijo = challenger
            with mlflow.start_run(
                run_name=f"{model_name}_cfg_{config_id}",
                nested=True
            ):
                # 1. loggear hiperparámetros
                mlflow.log_params({
                    "model_name": model_name,
                    "config_id": config_id,
                    **param_dict
                })

                # 2. definir modelo y pipeline completo
                model = EstimatorClass(**param_dict)

                full_pipeline = Pipeline(steps=[
                    ("features", feature_pipeline),
                    ("model", model),
                ])

                # 3. entrenar y evaluar
                full_pipeline.fit(X_train, y_train)
                y_pred = full_pipeline.predict(X_val)
                rmse = np.sqrt(mean_squared_error(y_val, y_pred))
                print(f"→ RMSE validación: {rmse:0.4f}")

                # 4. loggear métricas de evaluación
                mlflow.log_metric("rmse_val", rmse)

                # 5. registrar el modelo challenger (guardado en artifacts)
                mlflow.sklearn.log_model(
                    full_pipeline,
                    artifact_path="model",
                )

                # 6. guardar resultados en memoria
                resultados.append({
                    "model": model_name,
                    "config_id": config_id,
                    "params": param_dict,
                    "rmse_val": rmse,
                    "run_id": mlflow.active_run().info.run_id,
                })

                # 7. actualizar champion en memoria
                if rmse < best_rmse:
                    best_rmse = rmse
                    best_model_name = model_name
                    best_config = param_dict
                    best_pipeline = full_pipeline

    # --- fin del loop: registramos el CHAMPION en el run padre ---
    elapsed = time.time() - start_time
    mlflow.log_metric("best_rmse", best_rmse)
    mlflow.log_metric("training_total_time_sec", elapsed)

    mlflow.set_tag("champion_model_name", best_model_name)
    mlflow.set_tag("pipeline_version", "v1")

    # Modelo campeón registrado como tal
    mlflow.sklearn.log_model(
        best_pipeline,
        artifact_path="champion_model",
        registered_model_name="ventas_champion_model",
    )

print("\n=== Mejor modelo encontrado ===")
print("Modelo:", best_model_name)
print("Hiperparámetros:", best_config)
print("RMSE validación:", best_rmse)


Entrenando linear_regression - config 1 con params: {'fit_intercept': True, 'positive': False}


2025/11/28 20:56:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


→ RMSE validación: 0.0000


2025/11/28 20:56:50 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run linear_regression_cfg_1 at: http://127.0.0.1:8080/#/experiments/721817035780238080/runs/55345e947ac04d72870b83424cf6e4eb
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/721817035780238080

Entrenando linear_regression - config 2 con params: {'fit_intercept': False, 'positive': False}


...

→ RMSE validación: 749.0273


2025/11/28 21:01:19 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/28 21:01:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run knn_cfg_3 at: http://127.0.0.1:8080/#/experiments/721817035780238080/runs/9e221ff4b9604b5b833f6386e74df48b
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/721817035780238080


2025/11/28 21:01:35 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'ventas_champion_model'.
2025/11/28 21:01:36 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ventas_champion_model, version 1
Created version '1' of model 'ventas_champion_model'.


🏃 View run tuning_modelos_ventas at: http://127.0.0.1:8080/#/experiments/721817035780238080/runs/2d4a132d02444cd685d9ef554f26be6e
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/721817035780238080

=== Mejor modelo encontrado ===
Modelo: linear_regression
Hiperparámetros: {'fit_intercept': True, 'positive': True}
RMSE validación: 1.1699205024482729e-12


### Resumen de entrenamiento y selección de modelo

- Se utilizó el dataset `feature_exploration_scaled.csv` con la variable objetivo **SALES**.
- Se realizó una partición **secuencial 80% / 20%** para entrenamiento y validación,
  respetando el orden temporal de las observaciones.
- Se cargó el pipeline de ingeniería de características `feature_pipeline.pkl`,
  que incorpora imputación, codificación, tratamiento de outliers, transformaciones
  y escalado de variables, incluyendo banderas para tienda y producto.
- Se evaluaron **5 modelos de regresión** (LinearRegression, Ridge, RandomForest,
  GradientBoosting y KNN) con **3 configuraciones de hiperparámetros cada uno**
  (15 entrenamientos en total).
- La métrica de comparación fue el **RMSE** sobre el conjunto de validación (20% final).
- El modelo con menor RMSE se seleccionó como **modelo ganador** y se reentrenó
  con el 100% de los datos disponibles.
- Finalmente, se guardó el **pipeline completo (preprocesamiento + modelo ganador)**
  en el archivo `full_sales_forecast_pipeline.pkl`, que será utilizado en
  `05_inference_calculation.ipynb` para generar predicciones sobre nuevos datos.